# 📺 YouTube Analytics & Audience Engagement in Germany

**Ironhack Data Analytics — Module 1 Project: Data Wrangling and Retrieval**

**Team:** Pariya & Zahid

---

## 🎯 Project Overview

**Main question:** *What drives engagement on trending YouTube videos in Germany?*

| # | Sub-question | Hypothesis |
|---|---|---|
| 1 | Does **video length** affect engagement? | Shorter videos get more likes & comments per view |
| 2 | Which **category** gets the most engagement? | Music has the highest like-rate |
| 3 | Does **publish time** matter? | Videos published on weekends get more views |
| 4 | How do **Shorts** compare to long videos? | *to be defined* |

## 📦 Data Sources

| # | Source | Type | Period | What it gives us |
|---|---|---|---|---|
| 1 | [YouTube Data API v3](https://developers.google.com/youtube/v3) | API | Snapshot of one day | Current trending videos in Germany (200), incl. **duration** |
| 2 | [YouTube Trending Video Dataset (Kaggle)](https://www.kaggle.com/datasets/rsrishav/youtube-trending-video-dataset) | CSV | Aug 2020 – Apr 2024 | Historical trending videos in Germany + category names (JSON) |

**Plan:** compare trending videos **then (2020–2024)** vs. **now**.

---
## 1. Setup

Import the libraries we need.

In [1]:

import requests
import pandas as pd
from getpass import getpass

### 🔑 API key

The key is entered with `getpass`, so it is **never shown or saved in the notebook** and never ends up on GitHub.

> Each team member uses **their own** API key (Google Cloud Console → YouTube Data API v3 → Credentials → API key).

In [2]:
api_key = getpass("Enter your YouTube API key: ")

---
## 2. Data Source 1 — YouTube Data API

### 2.1 Understand the response (one test request)

We use the `videos` endpoint with these parameters:

| Parameter | Value | Meaning |
|---|---|---|
| `part` | `snippet,statistics,contentDetails` | Which information we want (title/channel, views/likes, duration) |
| `chart` | `mostPopular` | Trending videos |
| `regionCode` | `DE` | Germany |
| `maxResults` | `50` | Max. number of videos per request (= per page) |
| `key` | `api_key` | Our API key (variable, not a string!) |

In [3]:
url = "https://www.googleapis.com/youtube/v3/videos"

params = {
    "part": "snippet,statistics,contentDetails",
    "chart": "mostPopular",
    "regionCode": "DE",
    "maxResults": 50,
    "key": api_key
}

response = requests.get(url, params=params)
print("Status code:", response.status_code)   # 200 = success

data = response.json()
data.keys()

Status code: 200


dict_keys(['kind', 'etag', 'items', 'nextPageToken', 'pageInfo'])

In [4]:
data["pageInfo"]

{'totalResults': 200, 'resultsPerPage': 50}

**What we found:**
- `items` → a **list** of videos (one dictionary per video) ✅ this is what we need
- `pageInfo` → `totalResults: 200` = there are **200** trending videos, but one request gives max. **50** (= one page)
- `nextPageToken` → the "key" to the next page 🔑

### 2.2 Look at one video

We take **one** video to find the path to each field.

In [5]:
first = data["items"][0]
first.keys()

dict_keys(['kind', 'etag', 'id', 'snippet', 'contentDetails', 'statistics'])

Each video has 3 main parts:

| Part | Fields we use |
|---|---|
| `snippet` | `title`, `channelTitle`, `categoryId`, `publishedAt` |
| `statistics` | `viewCount`, `likeCount`, `commentCount` |
| `contentDetails` | `duration`, `caption` |

> ℹ️ `duration` uses the ISO 8601 format, e.g. `PT2M43S` = 2 minutes 43 seconds → converted to seconds during cleaning.

### 2.3 Function: fetch all 200 videos (pagination & error handling)

200 videos ÷ 50 per page = **4 requests**. Every page gives us the key (`nextPageToken`) for the next page:

| Request | We send | We get |
|---|---|---|
| 1 | normal params | videos 1–50 + key for page 2 |
| 2 | params + `pageToken` | videos 51–100 + key for page 3 |
| 3 | params + `pageToken` | videos 101–150 + key for page 4 |
| 4 | params + `pageToken` | videos 151–200 |

We put this in a **reusable function** `fetch_trending_videos()`:
- `region` and `pages` can be changed (e.g. `region="FR"` for France)
- **Error handling:** if a request fails (e.g. wrong API key or quota exceeded), the function prints the error and stops instead of crashing

In [6]:
def fetch_trending_videos(api_key, region="DE", pages=4):
    """Fetch trending YouTube videos for a country (50 per page)."""
    all_items = []
    next_page = None

    for page in range(pages):
        params = {
            "part": "snippet,statistics,contentDetails",
            "chart": "mostPopular",
            "regionCode": region,
            "maxResults": 50,
            "key": api_key
        }
        if next_page:
            params["pageToken"] = next_page

        response = requests.get(url, params=params)

        # error handling: stop if the request failed
        if response.status_code != 200:
            print("Error:", response.status_code)
            break

        data = response.json()

        for item in data["items"]:
            all_items.append(item)

        next_page = data.get("nextPageToken")

    return all_items


all_items = fetch_trending_videos(api_key)
len(all_items)   # should be 200


200

### 2.4 Extract the fields into a DataFrame

We loop over every video, build a dictionary with the fields we need, and collect them in a list.

> ⚠️ Some channels **hide likes or comments** → those keys are missing and `video['statistics']['likeCount']` raises a `KeyError`.
> We use `.get()` instead, which returns `None` (→ NaN) if the key doesn't exist.

In [7]:
def extract_video_info(items):
  """Extract the fields we need from each video → DataFrame."""
  video_list = []

  for video in items:
      video_info = {
          "video_id": video["id"],
          "title": video["snippet"]["title"],
          "channel": video["snippet"]["channelTitle"],
          "category": video["snippet"]["categoryId"],
          "published_at": video["snippet"]["publishedAt"],
          "duration": video["contentDetails"]["duration"],
          "views": video["statistics"].get("viewCount"),
          "likes": video["statistics"].get("likeCount"),
          "comments": video["statistics"].get("commentCount"),
          "caption": video["contentDetails"]["caption"]
      }
      video_list.append(video_info)
  return pd.DataFrame(video_list)
df_api=extract_video_info(all_items)
df_api.shape


(200, 10)

In [8]:


print(df_api.shape)
df_api.head()

(200, 10)


,video_id,title,channel,category,published_at,duration,views,likes,comments,caption
0,mw3kSNIxjqo,Taylor Swift - Patient Zero (Official Music Vi...,Taylor Swift,10,2026-09-29T22:00:19Z,PT5M13S,5984916,227355,10785,false
1,XoAi0d0VUgo,GEKLAUTE WEIHNACHTEN Teaser Trailer German Deu...,KinoCheck,24,2026-09-30T13:14:13Z,PT59S,89526,670,90,false
2,jS_PgXx4JqI,ICH BIN FÜR IMMER IM STRONGHOLD GEFANGEN (XXL ...,BastiGHG,20,2026-09-30T18:27:48Z,PT1H17M34S,267313,9782,314,false
3,akyKrBKGRio,Movin' to the Sun (Soul Edition),Simo Sent - Topic,10,2026-09-30T03:03:55Z,PT3M57S,124847,2588,0,false
4,Ky5NkR51W54,The Man with the Bag | Offizieller Teaser | Pr...,Amazon Prime Video Deutschland,24,2026-09-30T13:00:41Z,PT42S,9203,55,3,false


### 2.5 Save the raw data (once!)

Trending videos **change every day**, so we save one snapshot and work with it from now on.

> 🔒 After saving once, this line is commented out with `#`, so running the notebook again doesn't overwrite our raw data.

In [ ]:
#df_api.to_csv("youtube_trending_de_api_raw.csv", index=False)

---
## 3. Cleaning — API data

From here on, we work with the **saved CSV** instead of calling the API again.
This way the data stays the same every time we run the notebook.

### 3.1 Load the raw data and check the data types

In [10]:
df = pd.read_csv(FOLDER + "youtube_trending_de_api_raw.csv")
df.info()

NameError: name 'FOLDER' is not defined

**What we found:**
- `read_csv` already converted `views`, `comments`, `category` to **numbers** and `caption` to **bool** ✅
- `likes` is **float** and has missing values → some channels hide likes (`NaN` forces float)
- `published_at` and `duration` are still **text** → need to be converted

### 3.2 Missing values & duplicates

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.isnull().sum()

,0
video_id,0
title,0
channel,0
category,0
published_at,0
duration,0
views,0
likes,1
comments,0
caption,0


**What we found:**
- **Missing values:** only `likes` has **1 missing value**. This channel has **hidden its likes**, so the API didn't send `likeCount` for this video.
- **Duplicates:** **0** duplicate rows. Each video appears only once, because the API data is a single-day snapshot.

**Decision:** we keep the missing value as **NaN** and do **not** replace it with 0.
The video does have likes, they are just hidden. Replacing them with 0 would be **wrong data** and would pull down the average. Pandas ignores NaN automatically when calculating `mean()`, so the analysis still works.

### 3.3 Rename columns
We rename `category` → `category_id` to match the **shared column format** with the Kaggle data (so both tables can be combined later).

In [ ]:
df = df.rename(columns={"category": "category_id"})
df.columns

Index(['video_id', 'title', 'channel', 'category_id', 'published_at',
       'duration', 'views', 'likes', 'comments', 'caption'],
      dtype='object')

### 3.4 Convert data types
- `published_at` was text → converted to **datetime** with `pd.to_datetime()`
- The API sends time in **UTC** → converted to **German time** (`Europe/Berlin`), because we analyse videos trending in Germany

In [ ]:
def convert_published_at_to_datetime(df):
    df["published_at"] = pd.to_datetime(df["published_at"])
    return df

df = convert_published_at_to_datetime(df)
df["published_at"].dtype

datetime64[ns, UTC]

In [ ]:
def convert_to_german_timezone(df):
    df["published_at"] = df["published_at"].dt.tz_convert("Europe/Berlin")
    return df

df = convert_to_german_timezone(df)
df["published_at"].head()

,published_at
0,2026-09-28 17:00:06+02:00
1,2026-09-28 13:03:03+02:00
2,2026-09-28 16:00:36+02:00
3,2026-09-28 12:00:11+02:00
4,2026-09-28 16:06:41+02:00


### 3.5 New columns: weekday & hour


In [ ]:
def extract_publish_weekday(df):
    df['publish_weekday'] = df['published_at'].dt.day_name()
    return df

df = extract_publish_weekday(df)
df['publish_weekday'].head()

,publish_weekday
0,Monday
1,Monday
2,Monday
3,Monday
4,Monday


In [ ]:
def extract_publish_hour(df):
    df['publish_hour'] = df['published_at'].dt.hour
    return df

df = extract_publish_hour(df)
df['publish_hour'].head()

,publish_hour
0,17
1,13
2,16
3,12
4,16


In [ ]:
df['publish_weekday'].value_counts()

,count
publish_weekday,
Monday,161
Sunday,20
Tuesday,10
Saturday,5
Wednesday,2
Thursday,1
Friday,1


**Insight:** 161 of 200 videos (80%) were published on **Monday**, because the data was collected on a Tuesday and trending mostly shows **very recent** videos.
→ The API data is **not suitable** for comparing weekdays (strong bias toward the day before collection).
→ Research question 3 (publish time) will be answered with the **Kaggle data** (3.5 years, all weekdays covered).

### 3.6 duration
`duration` was in ISO 8601 format (e.g. `PT2M43S` = 2 min 43 sec) → converted to **seconds** (`duration_sec`) with `pd.to_timedelta()`, so we can calculate with it.

In [ ]:

def add_duration_sec(df):
    """Convert ISO 8601 duration (e.g. 'PT2M43S') to seconds."""
    df["duration_sec"] = pd.to_timedelta(df["duration"]).dt.total_seconds()
    return df

df = add_duration_sec(df)
df[["duration", "duration_sec"]].head()

,duration,duration_sec
0,PT2M43S,163.0
1,PT2M25S,145.0
2,PT14M51S,891.0
3,PT4M34S,274.0
4,PT2M9S,129.0


In [ ]:
df['duration_sec'].describe()

,duration_sec
count,200.000000
mean,2761.015000
std,4000.672703
min,34.000000
25%,916.500000
50%,1584.500000
75%,2915.500000
max,35172.000000


In [ ]:
df[df["duration_sec"]==df['duration_sec'].max()]

,video_id,title,channel,category_id,published_at,duration,views,likes,comments,caption,publish_weekday,publish_hour,duration_sec
115,GtoOqFzG0rY,MOJ NAJTEZI CHALLENGE DO SAD,Nugato,20,2026-09-29 07:31:57+02:00,PT9H46M12S,165171,4619.0,12,False,Tuesday,7,35172.0


**Sanity check:**
- Shortest video: **34 sec**, longest: **≈9h 46min** (a gaming live-stream by *Nugato*)
- The average length is **≈46 min**, but the middle value (median) is only **≈26 min**.
  → A few **very long videos** pull the average up, so the average is misleading.
- **Decision:** we **keep** the long videos (they are real, not mistakes), but we use the **median** instead of the average.

In [ ]:
df.head()

,video_id,title,channel,category_id,published_at,duration,views,likes,comments,caption,publish_weekday,publish_hour,duration_sec
0,s_JQEVcu9yI,Bujar Mustafa - Belaja,BM Official,10,2026-09-28 17:00:06+02:00,PT2M43S,139332,2760.0,100,False,Monday,17,163.0
1,JZZz8m0N3t8,ASTERIX UND DAS KÖNIGREICH NUBIEN Trailer 2 Ge...,KinoCheck,24,2026-09-28 13:03:03+02:00,PT2M25S,149592,1363.0,139,False,Monday,13,145.0
2,wFDj6uL_iK8,MIMIC PARTY IN TEAMS,LUCREW,24,2026-09-28 16:00:36+02:00,PT14M51S,252261,10704.0,294,False,Monday,16,891.0
3,6Ws4ydd85wU,MEHNERSMOOS - Intro,Mehnersmoos,10,2026-09-28 12:00:11+02:00,PT4M34S,52398,3631.0,287,False,Monday,12,274.0
4,uvDjQokpKUo,THE MORTUARY ASSISTANT Trailer German Deutsch ...,KinoCheck Heimkino,24,2026-09-28 16:06:41+02:00,PT2M9S,23309,164.0,15,False,Monday,16,129.0


### 3.7 Engagement rates
The number of likes alone is misleading (popular videos naturally get more likes).
→ We calculate **engagement rates** in **%**:
- `like_rate` = likes / views × 100
- `comment_rate` = comments / views × 100

In [ ]:
def convert_to_numeric(dataframe, column_name, dtype):
    dataframe[column_name] = pd.to_numeric(dataframe[column_name], errors='coerce').astype(dtype)
    return dataframe

def calculate_engagement_rates(dataframe):
    dataframe = convert_to_numeric(dataframe, 'likes', float)
    dataframe = convert_to_numeric(dataframe, 'views', int)
    dataframe = convert_to_numeric(dataframe, 'comments', float)

    dataframe['like_rate'] = dataframe['likes'] / dataframe['views'] * 100
    dataframe['comment_rate'] = dataframe['comments'] / dataframe['views'] * 100
    return dataframe

df = calculate_engagement_rates(df)
df[['like_rate', 'comment_rate']].head()

,like_rate,comment_rate
0,1.980880,0.071771
1,0.911145,0.092919
2,4.243224,0.116546
3,6.929654,0.547731
4,0.703591,0.064353


In [ ]:
df['like_rate'].isnull().sum()

np.int64(1)

### 3.8 Category names (from Kaggle JSON)
- The API only gives a category **number** (e.g. `10`) → we add the **category names** from `DE_category_id.json` (Kaggle)
- `category_id` in the JSON was **text** → converted to **int** so it matches our data before merging
- `how="left"` keeps **all 200 videos**, even if a category is not found

In [ ]:
import json

with open("DE_category_id.json") as f:
    categories = json.load(f)

categories.keys()

dict_keys(['kind', 'etag', 'items'])

In [ ]:
categories['items'][0]

{'kind': 'youtube#videoCategory',
 'etag': 'IfWa37JGcqZs-jZeAyFGkbeh6bc',
 'id': '1',
 'snippet': {'title': 'Film & Animation',
  'assignable': True,
  'channelId': 'UCBR8-60-B28hp2BmDPdntcQ'}}

In [ ]:
categories['items'][0]['snippet']['title']

'Film & Animation'

In [ ]:
categories['items'][0]['id']

'1'

In [ ]:
video_category ={"category_id":categories['items'][0]['id'],"category_name":categories['items'][0]['snippet']['title']}
video_category

{'category_id': '1', 'category_name': 'Film & Animation'}

#### Merge category names

Our data only has the category **number** (`category_id`, e.g. `10`).
We add the category **name** (e.g. `Music`) from `cat_df` by merging both tables on `category_id`.

- Both `category_id` columns must have the **same type** (int), otherwise the merge doesn't match
- `how="left"` keeps **all 200 videos**, even if a category is not found

In [ ]:
def load_categories(path):
    """Read the category JSON and return a DataFrame with category_id (int) and category_name."""
    with open(path) as f:
        categories = json.load(f)
    category_list = [
        {"category_id": int(cat["id"]), "category_name": cat["snippet"]["title"]}
        for cat in categories["items"]
    ]
    return pd.DataFrame(category_list)


def add_category_names(df, cat_df):
    """Add category names by merging on category_id (left join keeps all videos)."""
    df["category_id"] = df["category_id"].astype(int)
    return df.merge(cat_df, on="category_id", how="left")


cat_df = load_categories(FOLDER + "DE_category_id.json")
df = add_category_names(df, cat_df)

print(df.shape)
print(df["category_name"].isnull().sum())
df[["category_id", "category_name"]].head()

(200, 16)
0


,category_id,category_name
0,10,Music
1,24,Entertainment
2,24,Entertainment
3,10,Music
4,24,Entertainment


**Result:** all 200 videos kept, **0** missing category names ✅

### 3.9 Add `period` column

We add a `period` column to show **where each row comes from**:
- API data → `"2026"` (current trending videos)
- Kaggle data → `"2020-2024"` (historical trending videos)

This way, after combining both datasets, we can compare **then vs. now** with `groupby("period")`.

In [ ]:
def add_period(df, period):
    df["period"] = period
    return df

df = add_period(df, "2026")

### 3.10 Drop columns we don't need

- `duration` → replaced by `duration_sec` (numbers instead of text)
- `caption` → not part of the shared format and not available in the Kaggle data

In [ ]:
def drop_unused_columns(df, columns):
    return df.drop(columns=columns)

df = drop_unused_columns(df, ["duration", "caption"])


### 3.11 Save the clean data

The clean API data is saved as `api_clean.csv` in our shared Drive folder.
This file will be combined with `kaggle_clean.csv` (Zahid) in the next notebook.

In [ ]:
df.to_csv("api_clean.csv", index=False)

In [ ]:
df.head()

,video_id,title,channel,category_id,published_at,views,likes,comments,publish_weekday,publish_hour,duration_sec,like_rate,comment_rate,category_name,period
0,s_JQEVcu9yI,Bujar Mustafa - Belaja,BM Official,10,2026-09-28 17:00:06+02:00,139332,2760.0,100.0,Monday,17,163.0,1.980880,0.071771,Music,2026
1,JZZz8m0N3t8,ASTERIX UND DAS KÖNIGREICH NUBIEN Trailer 2 Ge...,KinoCheck,24,2026-09-28 13:03:03+02:00,149592,1363.0,139.0,Monday,13,145.0,0.911145,0.092919,Entertainment,2026
2,wFDj6uL_iK8,MIMIC PARTY IN TEAMS,LUCREW,24,2026-09-28 16:00:36+02:00,252261,10704.0,294.0,Monday,16,891.0,4.243224,0.116546,Entertainment,2026
3,6Ws4ydd85wU,MEHNERSMOOS - Intro,Mehnersmoos,10,2026-09-28 12:00:11+02:00,52398,3631.0,287.0,Monday,12,274.0,6.929654,0.547731,Music,2026
4,uvDjQokpKUo,THE MORTUARY ASSISTANT Trailer German Deutsch ...,KinoCheck Heimkino,24,2026-09-28 16:06:41+02:00,23309,164.0,15.0,Monday,16,129.0,0.703591,0.064353,Entertainment,2026


---
## 🗒️ Notes & Next Steps

### Known limitations
- Trending videos are the **most successful** videos → results describe trending content, not all of YouTube
- API data is a **single-day snapshot (200 videos)**, Kaggle covers **3.5 years (~55k videos)** → compare **means and percentages**, not counts
- Kaggle data has **no duration** → questions about video length / Shorts use the API data only

---

## 🧹 Shared Cleaning Format (API + Kaggle)

Both datasets must end up with **the same column names and types**, so we can combine them with `concat`.

### Final columns

| Column | Type | API (Pariya) | Kaggle (Zahid) |
|---|---|---|---|
| `video_id` | string | `video_id` | `video_id` |
| `title` | string | `title` | `title` |
| `channel` | string | `channel` | rename `channelTitle` |
| `category_id` | int | rename `category` | rename `categoryId` |
| `category_name` | string | from `DE_category_id.json` | from `DE_category_id.json` |
| `published_at` | datetime | `published_at` | rename `publishedAt` |
| `publish_weekday` | string | from `published_at` | from `published_at` |
| `publish_hour` | int | from `published_at` | from `published_at` |
| `views` | int | `views` | rename `view_count` |
| `likes` | float (NaN if hidden) | `likes` | `likes` |
| `comments` | float (NaN if hidden) | `comments` | rename comment column (check with `df.columns`) |
| `like_rate` | float | `likes / views` | `likes / views` |
| `comment_rate` | float | `comments / views` | `comments / views` |
| `duration_sec` | int | from `duration` | ❌ not available |
| `period` | string | `"2026"` | `"2020-2024"` |

### Cleaning steps (same order for both)
1. **Load** the raw CSV and check `df.info()`, `df.isna().sum()`, `df.duplicated().sum()`
2. **Rename** columns to the final names above
3. **Convert types**: `published_at` → datetime, numbers → int/float
4. **Missing values**: hidden likes/comments stay **NaN** (not 0, because 0 would be wrong data)
5. **Duplicates**: Kaggle has the same video on several trending days → keep **one row per video** (the last trending day)
6. **New columns**: `publish_weekday`, `publish_hour`, `like_rate`, `comment_rate` (+ `duration_sec` for API)
7. **Category names**: merge with `DE_category_id.json` on `category_id`
8. **Add** the `period` column
9. **Save** the clean result: `api_clean.csv` / `kaggle_clean.csv` in the shared Drive folder

---

### Who does what
- **Pariya** → API data (this notebook)
- **Zahid** → Kaggle data (own notebook, same format)
- **Together** → combine both + EDA

### Next sections
1. Cleaning — API data (continue here)
2. Cleaning — Kaggle data
3. Combining (`concat` API + Kaggle)
4. EDA — answer the research questions
5. Conclusions